In [1]:
1651 +203 +283 +1749 +222 +1650 +1855 +276 +213 +1827 +1993 +235 +1711 +336 +353 +1827

16384

In [2]:
1895 +152 +196 +1862 +231 +1973 +1762 +241 +128 +1788 +1719 +147 +1955 +165 +230 +1940

16384

In [81]:
import numpy as np
import itertools
from numpy import kron

In [82]:
# --- FILE 1: the ideal GHZ state (complex vector) ---
state_ghz = np.loadtxt("states_ghz.txt", dtype=complex)

# --- FILE 2: the list of measurement settings (0=X,1=Y,2=Z) ---
settings = np.loadtxt("strings_ghz.txt", dtype=int)

# --- FILE 3: the counts for each setting (each row has 16 numbers) ---
counts = np.loadtxt("counts_ghz.txt", dtype=int)

num_settings = settings.shape[0]
assert counts.shape == (num_settings, 16)


In [83]:
rho_ghz = np.outer(state_ghz, state_ghz.conj())

In [85]:
I = np.array([[1,0],[0,1]])
X = np.array([[0,1],[1,0]])
Y = np.array([[0, -1j],[1j, 0]])
Z = np.array([[1,0],[0,-1]])

paulis = [X, Y, Z]

def pauli_operator_from_setting(setting_row):
    """Return the (2^n × 2^n) operator for a string like [0,2,1,0]."""
    op = 1
    for s in setting_row:
        op = kron(op, paulis[s])
    return op


We construct the eigenvalue of a 4-bit string:

In [86]:
def bitstrings(n):
    """Generate all bitstrings of length n as integers 0..2^n-1."""
    return [np.array(list(np.binary_repr(i, width=n)), dtype=int) for i in range(2**n)]

bs = bitstrings(4)

In [87]:
def eigenvalue(setting_row, b):
    """Eigenvalue of P = P1 ⊗ P2 ⊗ ... on bitstring b."""
    ev = 1
    for s, bit in zip(setting_row, b):
        # For X,Y,Z the measurement gives 0=+1 eigenvalue, 1=-1 eigenvalue
        ev = ev * (+1 if bit == 0 else -1)
    return ev

Expectation values for all measurement settings:

In [88]:
expectation_values = []

for k in range(num_settings):
    setting = settings[k]
    row_counts = counts[k]
    N = row_counts.sum()
    probs = row_counts / N

    # sum_b p(b) * eigenvalue(P, b)
    exp_val = 0
    for prob, b in zip(probs, bs):
        exp_val += prob * eigenvalue(setting, b)

    expectation_values.append(exp_val)

expectation_values = np.array(expectation_values)


Build the Pauli operators as matrices (for SDP):

In [89]:
pauli_ops = [pauli_operator_from_setting(s) for s in settings]

In [90]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import minimize
from scipy.linalg import norm
from scipy.linalg import expm, sinm, cosm
import random
import cvxpy as cp
import itertools
import random
from scipy.linalg import eigh


sx = np.array([[0, 1], [1, 0]])  
sz = np.array([[1, 0], [0, -1]])  
sy = np.array([[0, -1j], [1j, 0]])  
identity = np.eye(2)  
I = np.eye(2)
s_plus = np.array([[0, 1], [0, 0]])
s_minus = np.array([[0, 0], [1, 0]])

def kron_n(*matrices):
    result = np.array([[1.0]])  
    for matrix in matrices:
        result = np.kron(result, matrix)
    return result

def pauli_operator(pauli_string):
    # print pauli_string
    op_list = [identity] * len(pauli_string)  # initialize the operator list with identity matrices

    for i, char in enumerate(pauli_string):
        if char == 'X':
            op_list[i] = sx
        elif char == 'Y':
            op_list[i] = sy
        elif char == 'Z':
            op_list[i] = sz
        elif char == 'I':
            op_list[i] = identity

    return kron_n(*op_list)

def pauli_expectation(rho, pauli_string, tol=1e-8):
    P = pauli_operator(pauli_string)
    exp_value = np.trace(rho @ P)
    
    if np.abs(exp_value.imag) > tol:
        print(f"[Warning] Large imaginary part detected: {exp_value.imag} for Pauli string {pauli_string}")
    
    return exp_value.real

def random_pauli_strings(L, Ns, seed=None):
    
    pauli_strings = []  

    if seed is not None:
        #np.random.seed(seed)
        random.seed(seed)

    for _ in range(Ns):
        
        # pick randomly L elemnents from 'I', 'X', 'Y', 'Z' and make a string to append to pauli_strings
        string = ''.join(random.choices(['I', 'X', 'Y', 'Z'], k=L))
        pauli_strings.append(string)

    return pauli_strings

def unique_random_pauli_strings(L, Ns, seed=None):
    if seed is not None:
        random.seed(seed)

    paulis = ['I', 'X', 'Y', 'Z']
    all_strings = [''.join(p) for p in itertools.product(paulis, repeat=L)]

    if Ns > len(all_strings):
        raise ValueError(f"Requested {Ns} strings, but only {len(all_strings)} unique strings available.")

    return random.sample(all_strings, Ns)

def pauli_expectations(rho, strings):
   
    values = []

    for s in strings:
        expectation = pauli_expectation(rho, s)
        values.append(expectation)

    return np.array(values)

In [91]:
def xxz_hamiltonian(L, J, Delta):
    H = np.zeros((2**L, 2**L), dtype=complex)  
    # print H shape
    print(H.shape)
    for j in range(L - 1):
        
        op_list_x = [identity] * (L)
        # print op_list_x shape
        # print(op_list_x)

        op_list_x[j] = sx
        op_list_x[j+1] = sx

        H += J * kron_n(*op_list_x)

    for j in range(L - 1):
        op_list_y = [identity] * (L)
        op_list_y[j] = sy
        op_list_y[j+1] = sy
        H += J * kron_n(*op_list_y)

    for j in range(L - 1):
        op_list_z = [identity] * (L)
        op_list_z[j] = sz
        op_list_z[j+1] = sz
        H += J * Delta * kron_n(*op_list_z)

    return H

In [92]:
def bitstrings(n):
    return [np.array(list(np.binary_repr(i, width=n)), dtype=int)
            for i in range(2**n)]

bs = bitstrings(4)

def eigenvalue(b):
    return (-1)**(b.sum())

# Compute expectation values for all 60 settings
exp_vals = []

for k in range(num_settings):
    row = counts[k]
    shots = row.sum()
    probs = row / shots
    exp_val = sum(probs[i] * eigenvalue(bs[i]) for i in range(16))
    exp_vals.append(exp_val)

exp_vals = np.array(exp_vals)

In [93]:
def compute_epsilon(Ns, n_shots=10000, delta=0.003):
    K = Ns
    return np.sqrt(2 * np.log(2 * K / delta) / n_shots)

In [94]:
def sdp_most_mixed_state_relaxed(L, pauli_ops_subset, pauli_values_subset, epsilon, solver=cp.SCS):

    rho = cp.Variable((2**L, 2**L), hermitian=True)
    purity = cp.sum_squares(cp.abs(rho))
    objective = cp.Minimize(cp.real(purity))

    constraints = [cp.trace(rho) == 1, rho >> 0]

    for i, Oi in enumerate(pauli_ops_subset):
        oi = float(np.real(pauli_values_subset[i]))
        constraints.append(cp.real(cp.trace(rho @ Oi)) >= oi - epsilon)
        constraints.append(cp.real(cp.trace(rho @ Oi)) <= oi + epsilon)

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=solver, verbose=False)
        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] status: {problem.status}")
            return None
        return rho.value
    except Exception as e:
        print(f"[SDP Error] {e}")
        return None


In [95]:
def optimal_unitary(rho, H):
    # Diagonalize rho: descending eigenvalues
    r_evals, r_evecs = np.linalg.eigh(rho)
    r_sort_idx = np.argsort(-r_evals)
    r_evecs = r_evecs[:, r_sort_idx]

    # Diagonalize H: ascending eigenvalues
    H_evals, H_evecs = np.linalg.eigh(H)
    H_sort_idx = np.argsort(H_evals)
    H_evecs = H_evecs[:, H_sort_idx]

    # Build the unitary: U = sum_j |E_j><r_j|
    U = H_evecs @ r_evecs.conj().T
    return U

In [96]:
import cvxpy as cp

def sdp_minimization_relaxed(L, pauli_ops_subset, pauli_values_subset, U, H, epsilon=1e-8):
    rho = cp.Variable((2**L, 2**L), hermitian=True)

    objective = cp.Minimize(cp.real(cp.trace((H - U.conj().T @ H @ U) @ rho)))

    constraints = [cp.trace(rho) == 1, rho >> 0]

    for i, Oi in enumerate(pauli_ops_subset):
        oi = pauli_values_subset[i]
        constraints.append(cp.real(cp.trace(rho @ Oi)) >= oi - epsilon)
        constraints.append(cp.real(cp.trace(rho @ Oi)) <= oi + epsilon)

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.SCS, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] Problem status: {problem.status}")
            return np.inf, None

        return problem.value, rho.value
    
    except Exception as e:
        print(f"[SDP Error] Solver failed: {e}")
        return np.inf, None


In [97]:
L = 4
L = 4
J = 1.0
Delta = 0.5
H_xxz = xxz_hamiltonian(L, J, Delta)

(16, 16)


In [98]:
pauli_values = exp_vals.copy()        # length 60, float expectation values
pauli_ops    = [pauli_operator_from_setting(s) for s in settings]

In [99]:
# --- TRUE ERGOTROPY of rho_ghz under H_xxz ---

def true_ergotropy(rho, H):
    """
    Compute the exact ergotropy of a density matrix rho under Hamiltonian H.
    """
    # eigenvalues/vectors of rho
    p, _ = eigh(rho)
    # eigenvalues of H
    e, _ = eigh(H)

    p_sorted = np.sort(p)[::-1]    # descending
    e_sorted = np.sort(e)          # ascending

    E_initial = np.real(np.trace(rho @ H))
    E_minimal = np.sum(p_sorted * e_sorted)

    return E_initial - E_minimal

true_E = true_ergotropy(rho_ghz, H_xxz)
print(f"True ergotropy GHZ–XXZ: {true_E:.6f}")


True ergotropy GHZ–XXZ: 6.924344


## Randomizing the choice

In [79]:
# === COMPUTE MEDIAN + IQR ===

Ns_list = np.arange(1, num_settings+1)   # 1..60
num_realizations = 30
n_shots = counts[0].sum()                 # shot count from real data

medians = []
iqr_lower = []
iqr_upper = []

for Ns in Ns_list:
    values_Ns = []

    for r in range(num_realizations):
        np.random.seed(200 + 17*r + Ns)

        # subset (first Ns constraints)
        idx = np.arange(Ns)
        ops_subset  = [pauli_ops[i] for i in idx]
        vals_subset = pauli_values[idx]

        epsilon_val = compute_epsilon(Ns, n_shots)

        # --- First SDP ---
        rho_feas = sdp_most_mixed_state_relaxed(4, ops_subset, vals_subset, epsilon_val)
        if rho_feas is None:
            continue

        # --- Optimal unitary ---
        U_opt = optimal_unitary(rho_feas, H_xxz)

        # --- Second SDP ---
        e_lb, _ = sdp_minimization_relaxed(
            4, ops_subset, vals_subset, U_opt, H_xxz, epsilon_val
        )

        if np.isfinite(e_lb):
            values_Ns.append(e_lb)

    values_Ns = np.array(values_Ns)
    
    if len(values_Ns) == 0:
        medians.append(np.nan)
        iqr_lower.append(np.nan)
        iqr_upper.append(np.nan)
        continue

    q25, q50, q75 = np.percentile(values_Ns, [25,50,75])

    medians.append(q50)
    iqr_lower.append(q50 - q25)
    iqr_upper.append(q75 - q50)

medians = np.array(medians)
iqr_lower = np.array(iqr_lower)
iqr_upper = np.array(iqr_upper)

print("Finished computing statistics.")


Finished computing statistics.


## Not random choice

In [108]:
# === COMPUTE MEDIAN + IQR ===

Ns_list = np.arange(1, num_settings+1)  # 1..60
n_shots = counts[0].sum()               # shot count from real data

medians = []
iqr_lower = []
iqr_upper = []

for Ns in Ns_list:
    # subset (first Ns constraints)
    idx = np.arange(Ns)
    ops_subset  = [pauli_ops[i] for i in idx]
    vals_subset = pauli_values[idx]

    epsilon_val = compute_epsilon(Ns, n_shots)

    # --- First SDP ---
    rho_feas = sdp_most_mixed_state_relaxed(4, ops_subset, vals_subset, epsilon_val)
    if rho_feas is None:
        medians.append(np.nan)
        iqr_lower.append(np.nan)
        iqr_upper.append(np.nan)
        continue

    # --- Optimal unitary ---
    U_opt = optimal_unitary(rho_feas, H_xxz)

    # --- Second SDP ---
    e_lb, _ = sdp_minimization_relaxed(
        4, ops_subset, vals_subset, U_opt, H_xxz, epsilon_val
    )

    if np.isfinite(e_lb):
        # Only one realization, so median = value, IQR = 0
        medians.append(e_lb)
        iqr_lower.append(0)
        iqr_upper.append(0)
    else:
        medians.append(np.nan)
        iqr_lower.append(np.nan)
        iqr_upper.append(np.nan)

medians = np.array(medians)
iqr_lower = np.array(iqr_lower)
iqr_upper = np.array(iqr_upper)

print("Finished computing statistics.")


Finished computing statistics.


In [109]:
# =======================
#        PLOT
# =======================

import matplotlib.pyplot as plt
import pylab
import numpy as np

####--- Plot style (NO LATEX) ----####
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

# --- values ---
x = Ns_list
med = np.maximum(medians, 0)
low = np.maximum(medians - iqr_lower, 0)
up  = np.maximum(medians + iqr_upper, 0)
erg_true = true_E

# --- plot ---
fig, ax = plt.subplots()

ax.fill_between(x, low, up, color="#4C72B0", alpha=0.25)

ax.plot(
    x, med, 'o-', 
    color="#4C72B0", linewidth=1.2, markersize=3,
    label="Ergotropy lower bound"
)

ax.axhline(
    y=erg_true,
    linestyle='--', color="#EE4B2B", linewidth=1.2,
    label="True ergotropy"
)

ax.axhline(0, color='gray', linewidth=0.6)

ax.set_xlabel(r"$|\mathcal{I}|$")
ax.set_ylabel("Ergotropy lower bound")

ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontsize(9)

ax.legend(fontsize=8)
plt.tight_layout()

fname = "ergotropy_bounds_GHZ_xxz_realData_nomono.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()

print("Saved:", fname)


Saved: ergotropy_bounds_GHZ_xxz_realData_nomono.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_23249/2612923786.py:67: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## Trying to recover monotonicity 

In [104]:
# ============================================================
# COMPUTE LOWER BOUND WITH ORDERED CONSTRAINTS + UNITARY REUSE
# ============================================================

Ns_list = np.arange(1, num_settings+1)  # 1 .. 60
n_shots = counts[0].sum()               # number of shots (same for all settings)

medians = []
iqr_lower = []
iqr_upper = []

previous_lb = None          # store E_LB(N_s-1)
previous_U_opt = None       # store U_opt(N_s-1)

for Ns in Ns_list:

    idx = np.arange(Ns)   # deterministic: first Ns constraints
    ops_subset  = [pauli_ops[i]    for i in idx]
    vals_subset = pauli_values[idx]

    epsilon_val = compute_epsilon(Ns, n_shots)

    # ----------------------------------------------------
    # 1) FIRST SDP: feasible state
    # ----------------------------------------------------
    rho_feas = sdp_most_mixed_state_relaxed(4, ops_subset, vals_subset, epsilon_val)

    if rho_feas is None:
        # store NaNs to avoid breaking arrays
        medians.append(np.nan)
        iqr_lower.append(np.nan)
        iqr_upper.append(np.nan)
        print("rho_feas is None at the first round for Ns =", Ns)
        continue

    # ----------------------------------------------------
    # 2) Compute candidate U_opt from this Ns
    # ----------------------------------------------------
    U_candidate = optimal_unitary(rho_feas, H_xxz)

    # ----------------------------------------------------
    # 3) SECOND SDP: using new U_candidate
    # ----------------------------------------------------
    lb_candidate, _ = sdp_minimization_relaxed(
        4, ops_subset, vals_subset, U_candidate, H_xxz, epsilon_val
    )

    # ----------------------------------------------------
    # 4) UNITARY REUSE RULE:
    #    If lb_candidate > previous_lb (worse), REUSE old U
    # ----------------------------------------------------
    if previous_lb is not None and np.isfinite(lb_candidate):
        if lb_candidate < previous_lb:
            # REJECT candidate; use previous U_opt
            lb_reuse, _ = sdp_minimization_relaxed(
                4, ops_subset, vals_subset, previous_U_opt, H_xxz, epsilon_val
            )
            used_lb = lb_reuse
        else:
            # accept candidate
            used_lb = lb_candidate
            previous_U_opt = U_candidate
    else:
        # first value: accept by default
        used_lb = lb_candidate
        previous_U_opt = U_candidate

    previous_lb = used_lb

    # ----------------------------------------------------
    # Only one realization → median = value, IQR = 0
    # ----------------------------------------------------
    medians.append(used_lb)
    iqr_lower.append(0)
    iqr_upper.append(0)

medians = np.array(medians)
iqr_lower = np.array(iqr_lower)
iqr_upper = np.array(iqr_upper)

print("Finished ordered-constraint + unitary-reuse reconstruction.")


Finished ordered-constraint + unitary-reuse reconstruction.


In [105]:
# =======================
#        PLOT
# =======================

import matplotlib.pyplot as plt
import pylab
import numpy as np

####--- PRL-style (no LaTeX engine) ----####
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

x = Ns_list
med = np.maximum(medians, 0)
low = np.maximum(medians - iqr_lower, 0)
up  = np.maximum(medians + iqr_upper, 0)
erg_true = true_E

fig, ax = plt.subplots()

# IQR band
ax.fill_between(x, low, up, color="#4C72B0", alpha=0.25)

# Median
ax.plot(
    x, med, 'o-',
    color="#4C72B0", linewidth=1.2, markersize=3,
    label="Ergotropy lower bound"
)

# True line
ax.axhline(erg_true, linestyle='--', color="#EE4B2B", linewidth=1.2,
           label="True ergotropy")

# Zero
ax.axhline(0, color='gray', linewidth=0.6)

ax.set_xlabel(r"$|\mathcal{I}|$")
ax.set_ylabel("Ergotropy lower bound")

ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontsize(9)

ax.legend(fontsize=8)
plt.tight_layout()

fname = "ergotropy_bounds_GHZ_xxz_realData_ordered_reuseU.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()

print("Saved:", fname)


Saved: ergotropy_bounds_GHZ_xxz_realData_ordered_reuseU.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_23249/1882515675.py:66: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()
